# Sprint 2 — GPU Baselines (Colab/Kaggle T4)

**Runs in order:**
1. T5-A: BETO fine-tuned on Cardiff ES train, 3 epochs (definitive, replaces 1-epoch CPU result)
2. T5-B: XLM-RoBERTa-base fine-tuned on Cardiff ES train, 3 epochs
3. T4: Prompting baseline on Qwen3-1.7B — zero-shot + few-shot k=4, 8, 16

**Prerequisites:**
- Upload / clone the repo to `/content/AI_LAB` (or adjust `REPO_ROOT` below)
- Cardiff ES data will be re-downloaded if not present

**After the run:** Download the `results/baselines/` folder and copy the JSON files to your local repo.

In [ ]:
# Install dependencies
!pip install transformers peft trl bitsandbytes datasets evaluate scikit-learn pysentimiento accelerate pyyaml tqdm -q

In [ ]:
import sys, os
REPO_ROOT = '/content/AI_LAB'  # adjust if cloned elsewhere
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Load Cardiff ES splits
from src.data.load_data import load_cardiff_es, DataConfig

config = DataConfig(dataset_name='cardiffnlp/tweet_sentiment_multilingual', language='es')
splits = load_cardiff_es(config=config)
train_ds = splits['train']
val_ds   = splits['validation']
test_ds  = splits['test']
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## T5-A: BETO fine-tuned (3 epochs, definitive)

In [ ]:
import json
from pathlib import Path
from src.models.encoder_baseline import EncoderFineTuner
from src.models.evaluate import save_result

if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

beto = EncoderFineTuner('dccuchile/bert-base-spanish-wwm-cased', epochs=3)
train_time_beto = beto.train(train_ds, val_ds, output_dir='/tmp/beto_cardiff')

peak_vram_beto = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else None
result_beto = beto.evaluate(test_ds, train_time_s=train_time_beto, peak_vram_gb=peak_vram_beto)

out_path = Path('results/baselines/beto_cardiff_es.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open('w') as f:
    json.dump(result_beto.to_json_dict(), f, indent=2)
save_result(result_beto)

print(f'BETO F1 macro = {result_beto.f1_macro:.4f}')
print(f'Per-class: {result_beto.f1_per_class}')
print(f'Trainable params: {result_beto.trainable_params:,}')
print(f'Peak VRAM: {peak_vram_beto} GB | Train time: {train_time_beto:.1f} s')
print(f'Inference latency: {result_beto.inference_latency_ms:.2f} ms/example')

## T5-B: XLM-RoBERTa-base fine-tuned (3 epochs)

In [ ]:
if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

xlmr = EncoderFineTuner('xlm-roberta-base', epochs=3)
train_time_xlmr = xlmr.train(train_ds, val_ds, output_dir='/tmp/xlmr_cardiff')

peak_vram_xlmr = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else None
result_xlmr = xlmr.evaluate(test_ds, train_time_s=train_time_xlmr, peak_vram_gb=peak_vram_xlmr)

out_path = Path('results/baselines/xlmr_base_cardiff_es.json')
with out_path.open('w') as f:
    json.dump(result_xlmr.to_json_dict(), f, indent=2)
save_result(result_xlmr)

print(f'XLM-R F1 macro = {result_xlmr.f1_macro:.4f}')
print(f'Per-class: {result_xlmr.f1_per_class}')
print(f'Trainable params: {result_xlmr.trainable_params:,}')
print(f'Peak VRAM: {peak_vram_xlmr} GB | Train time: {train_time_xlmr:.1f} s')
print(f'Inference latency: {result_xlmr.inference_latency_ms:.2f} ms/example')

## T4: Prompting baseline — Qwen3-1.7B (zero-shot + few-shot k=4,8,16)

Expected runtime: ~30-60 min on T4 for the full test set (870 examples × 4 variants).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_ID = 'Qwen/Qwen3-1.7B'
print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_qwen.eval()
print('Model loaded.')

In [ ]:
from tqdm.auto import tqdm
from src.models.prompting import PromptingBaseline
from src.models.evaluate import compute_metrics, ExperimentResult, save_result, CostTracker

K_VALUES = [0, 4, 8, 16]
test_texts = test_ds['text']
test_labels = test_ds['label']
LABEL_NAMES = ('negative', 'neutral', 'positive')

for k in K_VALUES:
    print(f'\n--- Prompting k={k} ---')
    baseline = PromptingBaseline(k=k)
    examples = baseline.select_examples(train_ds, k=k, seed=42) if k > 0 else []

    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    # Run predict_batch with tqdm progress (wrap texts in tqdm)
    predictions, fallback_rate = baseline.predict_batch(
        tqdm(test_texts, desc=f'k={k}'),
        model_qwen, tokenizer,
        examples=examples,
        device=device,
    )
    elapsed = time.time() - t0
    peak_vram = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else None
    latency_ms = (elapsed / len(test_texts)) * 1000

    # Convert string predictions to int for metrics
    label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
    y_pred = [label2id[p] for p in predictions]
    metrics = compute_metrics(test_labels, y_pred, LABEL_NAMES)

    method = f'prompting_zeroshot' if k == 0 else f'prompting_fewshot_k{k}'
    result = ExperimentResult(
        experiment_name=f'{method}_Qwen3-1.7B',
        model_name=MODEL_ID,
        method=method,
        data_fraction='full',
        seed=42,
        f1_macro=metrics['f1_macro'],
        accuracy=metrics['accuracy'],
        f1_per_class=metrics['f1_per_class'],
        trainable_params=0,
        peak_vram_gb=peak_vram,
        train_time_s=0.0,
        inference_latency_ms=latency_ms,
        fallback_rate=fallback_rate,
        notes=(
            f'Prompting {method} on Qwen3-1.7B. Protocol: configs/prompting_protocol.yaml. '
            f'fallback_rate={fallback_rate:.3f}. Executed on Colab T4 GPU.'
        ),
    )

    # Save JSON
    fname = 'results/baselines/prompting_zeroshot_PENDING.json' if k == 0 else f'results/baselines/prompting_{k}shot_qwen3_1.7b.json'
    out_path = Path(fname)
    with out_path.open('w') as f:
        json.dump(result.to_json_dict(), f, indent=2)
    save_result(result)

    print(f'k={k} | F1 macro = {result.f1_macro:.4f} | fallback_rate = {fallback_rate:.3f} | latency = {latency_ms:.1f} ms/ex')

print('\nAll prompting variants done.')

In [ ]:
# Summary of all results
from src.models.evaluate import load_results
df = load_results()
cols = ['experiment_name', 'method', 'f1_macro', 'trainable_params',
        'peak_vram_gb', 'train_time_s', 'inference_latency_ms', 'fallback_rate']
print(df[cols].to_string(index=False))

## Download results

Run the cell below to zip `results/baselines/` and download it.
Then copy the JSON files to your local `results/baselines/` directory.

In [ ]:
!zip -r /content/sprint2_results.zip results/baselines/ results/all_results.csv
from google.colab import files
files.download('/content/sprint2_results.zip')